In [1]:
print("Hello, Agent")

Hello, Agent


In [2]:
%pip install -U openai python-dotenv

   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.3 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.3 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.3 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.3 MB ? eta -:--:--
   ---- ----------------------------------- 0.3/2.3 MB ? eta -:--:--
   --------- ------------------------------ 0.5/2.3 MB 335.7 kB/s eta 0:00:06
   --------- ------------------------------ 0.5/2.3 MB 335.7 kB/s eta 0:00:06
   --------- ------------------------------ 0.5/2.3 MB 335.7 kB/s eta 0:00:06
   ------------- -------------------------- 0.8/2.3 MB 372.9 kB/s eta 0:00:0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 1.2.1 requires openai<3.0.0,>=2.26.0, but you have openai 3.35.1 which is incompatible.
openai-agents 0.17.2 requires openai<3,>=2.26.0, but you have openai 3.35.1 which is incompatible.


In [6]:
import os
from openai import OpenAI

api_key = os.getenv("DEEPSEEK_API_KEY")

client = OpenAI(
    api_key=api_key,
    base_url="https://api.deepseek.com"
)

print("客户端初始化成功！")

客户端初始化成功！


In [7]:
response = client.chat.completions.create(
    model="deepseek-flash",
    messages=[
        {
            "role": "system",
            "content": "你是一名专业的 Python 开发导师，回答简洁清晰。"
        },
        {
            "role": "user",
            "content": "请用三句话解释什么是 AI Agent。"
        }
    ],
    extra_body={"thinking": {"type": "disabled"}},
    stream=False
)

print(response.choices[0].message.content)

1. AI Agent 是以大语言模型为“大脑”、能自主感知环境并决定下一步行动的智能程序。  
2. 它通过调用工具（搜索、代码执行、API 等）与外部世界交互，把“想”变成“做”。  
3. 与普通问答不同，它会规划步骤、循环执行、根据反馈调整，直到完成用户给定的目标。


注意：大模型 API 通常是无状态调用。
也就是说，你下一次请求如果不传入之前的对话历史，模型就不知道你上一轮聊了什么。
这个机制非常重要，后续实现 Agent Memory 时会用到。

In [8]:
print("模型：", response.model)

print("回答：")
print(response.choices[0].message.content)

print("\nToken 使用情况：")
print("输入 Token：", response.usage.prompt_tokens)
print("输出 Token：", response.usage.completion_tokens)
print("总 Token：", response.usage.total_tokens)

模型： deepseek-flash
回答：
1. AI Agent 是以大语言模型为“大脑”、能自主感知环境并决定下一步行动的智能程序。  
2. 它通过调用工具（搜索、代码执行、API 等）与外部世界交互，把“想”变成“做”。  
3. 与普通问答不同，它会规划步骤、循环执行、根据反馈调整，直到完成用户给定的目标。

Token 使用情况：
输入 Token： 26
输出 Token： 81
总 Token： 107


响应对象中的几个重要字段:
response
│
├── id
├── model
├── choices
│   └── [0]
│       ├── finish_reason
│       └── message
│           ├── role
│           └── content
│
└── usage
    ├── prompt_tokens
    ├── completion_tokens
    └── total_tokens

In [9]:
data = response.model_dump()

print(type(data))
print(data.keys())

<class 'dict'>
dict_keys(['id', 'choices', 'created', 'model', 'object', 'metadata', 'moderation', 'service_tier', 'system_fingerprint', 'usage'])


# TODO 1:
# 从 data 字典中取出模型名称


# TODO 2:
# 从 data 字典中取出模型回答文本
# 提示：观察 choices 的结构


# TODO 3:
# 使用 json.dumps() 将 data 转换成格式化的 JSON 字符串
# 要求：缩进 2 个空格，正确显示中文

In [10]:
model_name = data["model"]
print(model_name)

deepseek-flash


In [12]:
model_ans=data["choices"][0]["message"]["content"]
print(model_ans)

1. AI Agent 是以大语言模型为“大脑”、能自主感知环境并决定下一步行动的智能程序。  
2. 它通过调用工具（搜索、代码执行、API 等）与外部世界交互，把“想”变成“做”。  
3. 与普通问答不同，它会规划步骤、循环执行、根据反馈调整，直到完成用户给定的目标。


In [15]:
import json
json_str = json.dumps(
    data,
    ensure_ascii=False,
    indent=2
)
print(json_str)

{
  "id": "f631fe7b-615e-463d-84c2-0a5ca24c4410",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "1. AI Agent 是以大语言模型为“大脑”、能自主感知环境并决定下一步行动的智能程序。  \n2. 它通过调用工具（搜索、代码执行、API 等）与外部世界交互，把“想”变成“做”。  \n3. 与普通问答不同，它会规划步骤、循环执行、根据反馈调整，直到完成用户给定的目标。",
        "refusal": null,
        "role": "assistant",
        "annotations": null,
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1792411682,
  "model": "deepseek-flash",
  "object": "chat.completion",
  "metadata": null,
  "moderation": null,
  "service_tier": null,
  "system_fingerprint": "e72d3c7f663d3e1383b00b3884cb4501",
  "usage": {
    "completion_tokens": 81,
    "prompt_tokens": 26,
    "total_tokens": 107,
    "completion_tokens_details": null,
    "prompt_tokens_details": {
      "audio_tokens": null,
      "cache_write_tokens": null,
      "cached_tokens": 0,
      "image_tokens": null,
   

In [25]:
from openai import (
    AuthenticationError,
    RateLimitError,
    APIConnectionError,
    APITimeoutError,
    APIStatusError
)

def ask_llm(question,temperature=0.7):
    try:
        response = client.chat.completions.create(
            model="deepseek-chat",
            messages=[
                {"role": "system", "content": "你是一个AI助手"},
                {"role": "user", "content": question}
            ],
            temperature=temperature
        )

        ans=response.choices[0].message.content
        token=response.usage.total_tokens
        return ans,token

    except AuthenticationError:
        print("API Key 无效，请检查密钥")

    except RateLimitError:
        print("请求过于频繁，请稍后重试")

    except APITimeoutError:
        print("请求超时")

    except APIConnectionError:
        print("网络连接失败")

    except APIStatusError as e:
        print(f"API 请求失败，状态码：{e.status_code}")

    return None

In [26]:
result = ask_llm("什么是 RAG？",temperature=0.5)
answer,token=result
print(answer)
print(token)

RAG（Retrieval-Augmented Generation，检索增强生成）是一种将**信息检索**与**大语言模型生成**相结合的技术架构。

## 核心思想

传统大语言模型（如 GPT）的知识完全来自训练数据，存在两个问题：知识有截止日期、容易"幻觉"（编造事实）。RAG 的思路是：在模型生成回答之前，先从外部知识库中检索出相关内容，再把这些内容作为上下文交给模型，让模型"看着资料答题"。

## 工作流程

1. **索引（Indexing）**：将文档切分成片段，用嵌入模型转为向量，存入向量数据库。
2. **检索（Retrieval）**：用户提问时，把问题也转为向量，在数据库中做相似度搜索，找出最相关的若干文档片段。
3. **生成（Generation）**：将检索到的片段与原始问题一起拼成提示词，交给大语言模型生成最终回答。

## 主要优势

- **减少幻觉**：回答有据可查，基于检索到的真实文档
- **知识可更新**：更新知识库即可，无需重新训练模型
- **可溯源**：能给出引用来源，便于核查
- **成本低**：比微调模型便宜得多，适合企业私有知识场景

## 典型应用

企业知识库问答、客服机器人、法律/医疗文档检索、代码助手等。RAG 是目前让大模型"接入私有数据"最主流的工程方案之一。
329


In [28]:
ans,tokens=ask_llm("deepseek的api怎么调用？")
print(ans,tokens)

# DeepSeek API 调用指南

DeepSeek 的 API **完全兼容 OpenAI 接口格式**，所以调用方式非常简单。以下是详细说明。

## 1. 获取 API Key

1. 访问 [DeepSeek 开放平台](https://platform.deepseek.com/)
2. 注册/登录后，进入「API Keys」页面
3. 创建新的 API Key 并保存（只显示一次）

## 2. 基本信息

| 项目 | 值 |
|------|-----|
| Base URL | `https://api.deepseek.com` |
| 兼容路径 | `https://api.deepseek.com/v1`（也可用） |
| 常用模型 | `deepseek-chat`（V3）、`deepseek-reasoner`（R1） |
| 认证方式 | `Authorization: Bearer <API_KEY>` |

## 3. 调用示例

### Python（使用 openai SDK，推荐）

```python
from openai import OpenAI

client = OpenAI(
    api_key="sk-xxxxxxxxxxxxxxxx",  # 你的 API Key
    base_url="https://api.deepseek.com"
)

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "你是一个有用的助手"},
        {"role": "user", "content": "你好，介绍一下你自己"}
    ],
    stream=False
)

print(response.choices[0].message.content)
```

安装依赖：`pip install openai`

### 使用 requests 直接调用

```python
import requests

url = "https://api.deepseek.com/chat/complet